In [1]:
# NAME: Grazioso Salvare Dashboard
# DESCRIPTION: Script to create a Jupyter Dash dashboard for a database
# AUTHOR: Joshua Pickle
# LAST EDITED: 2/15/2026

# Used ChatGPT for some bug-fixing. Code is my own.

# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the imports necessary for Flask-Login for security purposes
import flask
from flask_login import UserMixin, current_user, login_user, LoginManager, login_required, logout_user

# Import user manager
from Users_Python_Module import UserManager

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html, callback, ctx, no_update
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure the plotting routines
import pandas as pd

# Allow access to Flask server (for Flask-Login)
server = flask.Flask(__name__)

# Import animal shelter
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# Use the username and password and CRUD Python module name
username = 'aacuser'
password = 'CS499Pass!'

# Connect to users database via User Manager
users = UserManager(username, password)

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# Class read method must support return of list object and accept projection json input
# Sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# Get Grazioso Salvare's logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#########################
# Configure Pages
#########################

# Set up app configuration, suppress callback exceptions to prevent annoying error
# (per the suggestion of the error)
app = JupyterDash(__name__, server = server, suppress_callback_exceptions = True)

# Show the displayed page
app.layout = html.Div([
    dcc.Location(id='url'),
    html.Div(id='page-content'),

    # Remember the number of times the "Logout" button has been pressed
    dcc.Store(id = 'logout-clicks', data = 1),
    # Remember the number of times the "Create Table Entry" button has been pressed
    dcc.Store(id = 'create-table-entry-clicks', data = 1),
    # Remember the number of times the "Update Table Entry" button has been pressed
    dcc.Store(id = 'update-table-entry-clicks', data = 1),
    # Remember the number of times the "Delete Table Entry" button has been pressed
    dcc.Store(id = 'delete-table-entry-clicks', data = 1),
    # Remember the number of times the "Cancel" button from the create menu has been pressed
    dcc.Store(id = 'create-cancel-clicks', data = 1),
    # Remember the number of times the "Cancel" button from the update menu has been pressed
    dcc.Store(id = 'update-cancel-clicks', data = 1),
    # Remember the number of times the "Cancel" button from the delete menu has been pressed
    dcc.Store(id = 'delete-cancel-clicks', data = 1),
    # Remember the number of times the "Create" button from the create menu has been pressed
    dcc.Store(id = 'create-clicks', data = 1),
    # Remember the number of times the "Update" button from the update menu has been pressed
    dcc.Store(id = 'update-clicks', data = 1),
    # Remember the number of times the "Delete" button from the create menu has been pressed
    dcc.Store(id = 'delete-clicks', data = 1)
])

# Determine which page should be displayed based on url path
@callback(
    Output('page-content', 'children'), 
    Input('url', 'pathname')
)
def display_page(pathname):
    if current_user.is_authenticated == True:
        if pathname == '/database':
            return database_layout
        elif pathname == '/create':
            return create_table_entry_layout
        elif pathname == '/update':
            return update_table_entry_layout
        elif pathname == '/delete':
            return delete_table_entry_layout
    return login_page
        
#########################
# Login Setup
#########################

# Create login manager to enable Flask-Login
login = LoginManager(server)

# Create secret key for login_user
server.config['SECRET_KEY'] = 'samplekey'

# User class to implement UserMixin for Flask-Login
class User(UserMixin):
    # Function to create a user
    def __init__(self, username):
        self.id = username

    # Function to get the id from a user
    def get_id(self):
        return self.id

#########################
# Login Layout / View
#########################

# Login page
login_page = html.Div([
    html.Div(
        className='col s12 m6',
        children = [
            html.Center(html.Img(
                src = 'data:image/png;base64,{}'.format(encoded_image.decode()), 
                style = {
                    'height': '10%', 
                    'width': '10%'
                }
            ))
        ]
    ),
    html.Center(html.B(html.H1('Database Login'))),
    html.Center(html.B(html.H2('Username'))),
    html.Center(dcc.Input(
        id = 'username-input', 
        placeholder = 'Input Username', 
        value = ''
    )),
    html.Center(html.B(html.H3(
        'Username Cannot Be Empty', 
        id = 'bad-username-text', 
        style={'color': 'red'}, 
        hidden = True
    ))),
    html.Center(html.B(html.H2('Password'))),
    html.Center(dcc.Input(
        id = 'password-input', 
        type = 'password', 
        placeholder = 'Input Password', 
        value = ''
    )),
    html.Center(html.B(html.H3(
        'Password Cannot Be Empty', 
        id = 'bad-password-text', 
        style = {'color': 'red'}, 
        hidden = True
    ))),
    html.Center(html.B(html.H3(
        'Incorrect username/password combination', 
        id = 'incorrect-signin-text', 
        style = {'color': 'red'}, 
        hidden = True
    ))),
    html.Center(html.B(html.H3(
        'User with username does already exists', 
        id = 'user-existence-text', 
        style = {'color': 'red'}, 
        hidden = True
    ))),
    html.Div(
        # Make child Divs go next to each other
        className='row',
        style = {
            'display': 'flex', 
            'justifyContent': 'center', 
            'marginTop': 25
        },
        children = [   
            html.Button(
                id = 'login-button', 
                children = 'Log In', 
                style = {'backgroundColor': '#67bed9'},
                n_clicks = 0
            ),
            html.Button(
                id = 'register-button', 
                children = 'Register',
                style = {'backgroundColor': '#d07597', 'marginLeft': 15},
                n_clicks = 0
            )
        ]
    )
])

#############################################
# Interaction Between Login Components / Controller
#############################################

# Set login page when logged out
login.login_view = login_page

# Implement ability to get current user
@login.user_loader
def load_user(id):
    if users.doesUserExist(id) == True:
        return User(id)
    else:
        return None

# Function to attempt to access the database (via login or register)
@callback(
    Output('url', 'pathname', allow_duplicate = True),
    Output('incorrect-signin-text', 'hidden'),
    Output('user-existence-text', 'hidden'),
    Input('login-button', 'n_clicks'),
    Input('register-button', 'n_clicks'),
    State('username-input', 'value'),
    State('password-input', 'value'),
    prevent_initial_call = True
)
def attempt_database_access(login_clicks, register_clicks, username, password):
    # Make the extension "/login" immediately
    if (login_clicks == 0 and register_clicks == 0):
        return '/login', True, True
    # Prevent access if username or password are empty
    elif username == '' or password == '':
        return no_update, True, True
    # Login
    if 'login-button' == ctx.triggered_id:
        # Check if username/password combination exists
        if users.isSignInCorrect(username, password):
            user = User(username)
            login_user(user)
            return '/database', True, True
        else:
            return no_update, False, True
    # Registration
    elif 'register-button' == ctx.triggered_id:
        # Check if user exists
        if not users.doesUserExist(username):
            # Create user
            users.createUser(username, password)
            user = User(username)
            login_user(user)
            return '/database', True, True
        # User already exists
        else:
            return no_update, True, False
    # Neither button was pressed, do nothing
    return no_update, no_update, no_update

# Function to check if the username and password text boxes have contents for the error texts
@callback(
    Output('bad-username-text', 'hidden'),
    Output('bad-password-text', 'hidden'),
    Input('login-button', 'n_clicks'),
    Input('register-button', 'n_clicks'),
    State('username-input', 'value'),
    State('password-input', 'value'),
    prevent_initial_call = True
)
def user_pass_texts_full(login_clicks, register_clicks, username, password):
    usernameFull = True
    passwordFull = True
    
    if username == '':
        usernameFull = False
    if password == '':
        passwordFull = False
        
    return usernameFull, passwordFull

#########################
# Dashboard Layout / View
#########################

# Database page
database_layout = html.Div([
    html.Div(
        # Make child Divs go next to each other
        className = 'row',
        style = {'display' : 'flex', 'height': '125px'},
        children = [
            # Display logo
            html.Div(
                className='col s12 m6',
                children = [
                    html.Img(
                        src = 'data:image/png;base64,{}'.format(encoded_image.decode()), 
                        style = {
                            'height': '100%', 
                            'width': '100%'
                        }
                    )
                ]
            ),
            # Show name text
            html.Center(html.Div(
                className = 'col s12 m6',
                children = [
                    html.B(html.H1('Grazioso Salvare Animals Dashboard', style = {'marginLeft': 425})),
                    html.B(html.H1('Written by Joshua Pickle', style = {'marginLeft': 425}))
                ]
            )),
            # Create logout button
            html.Div(
                className = 'col s12 m6',
                children = [
                    html.Button(
                        id = 'logout-button', 
                        children = 'Log Out',
                        style = {
                            'backgroundColor': '#d07597', 
                            'marginLeft': 425,
                            'marginTop': 50
                        },
                        n_clicks = 0
                    )
                ]
            )
        ],
    ),
    html.Hr(),
    html.Div(
        # Make child Divs go next to each other
        className = 'row',
        style = {'display' : 'flex'},
        children = [
            # Create radio buttons for choosing queries
            html.Div(
                className = 'col s12 m6',
                children = [
                    dcc.RadioItems(
                        id = 'filter-type',
                        options = [
                           {'label': 'Water Rescue', 'value': 'WaterRescue'},
                           {'label': 'Mountain Rescue', 'value': 'MtnRescue'},
                           {'label': 'Disaster Rescue', 'value': 'DisasterRescue'},
                           {'label': 'Reset', 'value': 'Default'},
                       ],
                       value = 'Default'
                    )
                ]
            ),
            # Create helper texts for explaining radio button functions
            html.Div(
                className='col s12 m6',
                children = [
                    html.U(html.P(
                        '?', 
                        title = 'Narrows search down to dogs fit for water rescue. Includes the following criteria:\nBreed is Laborator Retriever Mix, Chesapeake Bay Retriever, or Newfoundland\nSex upon outcome must be \"Intact Female\"\nAge upon outcome in weeks must be between 26 and 156', 
                        style = {
                            'marginTop': 0, 
                            'marginBottom': 0, 
                            'marginLeft': -23, 
                            'color': 'blue'
                        }
                    )),
                    html.U(html.P(
                        '?',
                        title = 'Narrows search down to dogs fit for mountain rescue. Includes the following criteria:\nBreed is German Shepherd, Alaskan Malamute, Old English Sheepdog, Siberian Husky, or Rottweiler\nSex upon outcome must be \"Intact Male\"\nAge upon outcome in weeks must be between 26 and 156',
                        style = {
                            'marginTop': 1, 
                            'marginBottom': 0, 
                            'marginLeft': 2, 
                            'color': 'blue'
                        }
                    )),
                    html.U(html.P(
                        '?',
                        title = 'Narrows search down to dogs fit for disaster rescue. Includes the following criteria:\nBreed is Doberman Pinscher, German Shepherd, Golden Retriever, Bloodhound, or Rottweiler\nSex upon outcome must be \"Intact Male\"\nAge upon outcome in weeks must be between 20 and 300',
                        style = {
                            'marginTop': 1, 
                            'marginBottom': 0, 
                            'marginLeft': -8, 
                            'color': 'blue'
                        }
                    )),
                    html.U(html.P(
                        '?', 
                        title = 'Resets the applied category, returning results to show all dogs (aside from categories applied on the table itself)',
                        style = {
                            'marginTop': 2, 
                            'marginBottom': 0, 
                            'marginLeft': -75, 
                            'color': 'blue'
                        }
                    ))
                ]
            ),
            # Create range search input boxes 
            html.Div(
                children = [
                    html.B(html.P(
                        'animal_id', 
                        style = {
                            'marginTop': 1,
                            'marginBottom': 0,
                            'marginLeft': 25
                        }
                    )),
                    html.Div(
                        dcc.Input(
                            id = 'animal-id-min', 
                            placeholder = 'Minimum', 
                            value = '',
                            style = {
                                'width': '50%',
                                'marginTop': 5,
                                'marginBottom': 0,
                                'marginLeft': 25
                            }
                        )
                    ),
                    html.Div(
                        dcc.Input(
                            id = 'animal-id-max', 
                            placeholder = 'Maximum', 
                            value = '',
                            style = {
                                'width': '50%',
                                'marginTop': 5,
                                'marginBottom': 0,
                                'marginLeft': 25
                            }
                        )
                    )
                ]
            ),
            html.Div(
                className='col s12 m6',
                children = [
                    html.B(html.P(
                        'breed', 
                        style = {
                            'marginTop': 1,
                            'marginBottom': 0
                        }
                    )),
                    html.Div(
                        dcc.Input(
                            id = 'breed-min', 
                            placeholder = 'Minimum', 
                            value = '',
                            style = {
                                'width': '50%',
                                'marginTop': 5,
                                'marginBottom': 0
                            }
                        )
                    ),
                    html.Div(
                        dcc.Input(
                            id = 'breed-max', 
                            placeholder = 'Maximum', 
                            value = '',
                            style = {
                                'width': '50%',
                                'marginTop': 5
                            }
                        )
                    )
                ]
            ),
            html.Div(
                className='col s12 m6',
                children = [
                    html.B(html.P(
                        'age_upon_outcome...', 
                        style = {
                            'marginTop': 1,
                            'marginBottom': 0
                        }
                    )),
                    html.Div(
                        dcc.Input(
                            id = 'age-upon-outcome-min', 
                            placeholder = 'Minimum', 
                            value = '',
                            style = {
                                'width': '50%',
                                'marginTop': 5,
                                'marginBottom': 0
                            }
                        )
                    ),
                    html.Div(
                        dcc.Input(
                            id = 'age-upon-outcome-max', 
                            placeholder = 'Maximum', 
                            value = '',
                            style = {
                                'width': '50%',
                                'marginTop': 5
                            }
                        )
                    )
                ]
            ),
            html.Div(
                className='col s12 m6',
                children = [
                    html.B(html.P(
                        'name', 
                        style = {
                            'marginTop': 1,
                            'marginBottom': 0
                        }
                    )),
                    html.Div(
                        dcc.Input(
                            id = 'name-min', 
                            placeholder = 'Minimum', 
                            value = '',
                            style = {
                                'width': '50%',
                                'marginTop': 5,
                                'marginBottom': 0
                            }
                        )
                    ),
                    html.Div(
                        dcc.Input(
                            id = 'name-max', 
                            placeholder = 'Maximum', 
                            value = '',
                            style = {
                                'width': '50%',
                                'marginTop': 5
                            }
                        )
                    )
                ]
            ),
            html.Div(
                className='col s12 m6',
                children = [
                    html.B(html.P(
                        'location_lat', 
                        style = {
                            'marginTop': 1,
                            'marginBottom': 0
                        }
                    )),
                    html.Div(
                        dcc.Input(
                            id = 'location-lat-min', 
                            placeholder = 'Minimum', 
                            value = '',
                            style = {
                                'width': '50%',
                                'marginTop': 5,
                                'marginBottom': 0
                            }
                        )
                    ),
                    html.Div(
                        dcc.Input(
                            id = 'location-lat-max', 
                            placeholder = 'Maximum', 
                            value = '',
                            style = {
                                'width': '50%',
                                'marginTop': 5
                            }
                        )
                    )
                ]
            ),
            html.Div(
                className='col s12 m6',
                children = [
                    html.B(html.P(
                        'location_long', 
                        style = {
                            'marginTop': 1,
                            'marginBottom': 0
                        }
                    )),
                    html.Div(
                        dcc.Input(
                            id = 'location-long-min', 
                            placeholder = 'Minimum', 
                            value = '',
                            style = {
                                'width': '50%',
                                'marginTop': 5,
                                'marginBottom': 0
                            }
                        )
                    ),
                    html.Div(
                        dcc.Input(
                            id = 'location-long-max', 
                            placeholder = 'Maximum', 
                            value = '',
                            style = {
                                'width': '50%',
                                'marginTop': 5
                            }
                        )
                    )
                ]
            ),
            html.Div(
                className='col s12 m6',
                children = [
                    html.Button(
                        id = 'submit-query-button', 
                        children = 'Submit Query',
                        style = {
                            'backgroundColor': '#67bed9',
                            'marginTop': 30
                        },
                        n_clicks = 0
                    )
                ]
            )
        ]
    ),
    html.Div(
        className = 'col s12 m6',
        children = [
            html.Center(html.B(html.H3(
                'Text used in value that can only take a number', 
                id = 'number-necessity-text', 
                style = {'color': 'red'}, 
                hidden = True
            )))
        ]
    ),
    html.Hr(),
    html.Div(
        className='row',
        style={'display' : 'flex'},
        children=[
            html.Div(
                className='col s12 m6',
                children = [
                    html.Button(
                        id = 'create-entry-button', 
                        children = 'Create Table Entry',
                        style = {
                            'backgroundColor': '#67bed9'
                        },
                        n_clicks = 0
                    )
                ]
            ),
            html.Div(
                className='col s12 m6',
                children = [
                    html.Button(
                        id = 'update-entry-button', 
                        children = 'Update Table Entry',
                        style = {
                            'backgroundColor': '#9c9ab8',
                            'marginLeft': 25
                        },
                        n_clicks = 0
                    )
                ]
            ),
            html.Div(
                className='col s12 m6',
                children = [
                    html.Button(
                        id = 'delete-entry-button', 
                        children = 'Delete Table Entry',
                        style = {
                            'backgroundColor': '#d07597',
                            'marginLeft': 25
                        },
                        n_clicks = 0
                    )
                ]
            )
        ]
    ),
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        column_selectable=False,
        row_selectable="single",
        row_deletable=False,
        selected_columns=[],
        selected_rows=[],
        page_action="native",
        page_current=0,
        page_size=10
    ),
    html.Br(),
    html.Hr(),
    # This sets up the dashboard so that the chart and the geolocation chart are side-by-side
    html.Div(
        className='row',
        style={'display' : 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
            ),
            html.Div(
                id='map-id',
                className='col s12 m6',
                children=[
                    dl.Map(
                        center=[56, 10], 
                        zoom=6, 
                        style={"height": "50vh"}, 
                        children=[
                            dl.TileLayer(id="base-layer-id")
                        ]
                    )
                ]
            )
        ]
    )
])

#############################################
# Interaction Between Database Components / Controller
#############################################

# Manipulate table data based on radio button + column range value input boxes 
@app.callback(
    Output('datatable-id','data'),
    Output('datatable-id', 'columns'),
    Output('number-necessity-text', 'hidden'),
    Input('submit-query-button', 'n_clicks'),
    State('filter-type', 'value'),
    State('animal-id-min', 'value'),
    State('animal-id-max', 'value'),
    State('breed-min', 'value'),
    State('breed-max', 'value'),
    State('age-upon-outcome-min', 'value'),
    State('age-upon-outcome-max', 'value'),
    State('name-min', 'value'),
    State('name-max', 'value'),
    State('location-lat-min', 'value'),
    State('location-lat-max', 'value'),
    State('location-long-min', 'value'),
    State('location-long-max', 'value')
)
def update_dashboard(submit_clicks, filter_type, id_min, id_max, breed_min,
                    breed_max, age_min, age_max, name_min, name_max, lat_min,
                    lat_max, long_min, long_max):
    # Declare dictionary for the read query
    readQuery = dict()

    # Declare variable for if there is a string that needs to be a number
    numberNecessityFailure = False
    
    # Apply radio button changes
    if (filter_type == "WaterRescue"):
        readQuery = {'animal_type': 'Dog', 'breed': {'$in': ['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland']},
                    'sex_upon_outcome': 'Intact Female', 'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}}
    elif (filter_type == "MtnRescue"):
        readQuery = {'animal_type': 'Dog', 'breed': {'$in': ['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog', 'Siberian Husky', 'Rottweiler']},
                    'sex_upon_outcome': 'Intact Male', 'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}}
    elif (filter_type == "DisasterRescue"):
        readQuery = {'animal_type': 'Dog', 'breed': {'$in': ['Doberman Pinscher', 'German Shepherd', 'Golden Retriever', 'Bloodhound', 'Rottweiler']},
                    'sex_upon_outcome': 'Intact Male', 'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}}

    # Remove any additional data outside of column range value input boxes
    # animal_id changes
    if id_min != '' or id_max != '':
        readQuery['animal_id'] = {}
        if id_min != '':
            readQuery['animal_id']['$gte'] = id_min
        if id_max != '':
            readQuery['animal_id']['$lte'] = id_max

    # breed changes
    if (breed_min != '' or breed_max != '') and 'breed' not in readQuery:
        readQuery['breed'] = {}
    # breed_min changes
    if breed_min != '':
        if '$in' in readQuery['breed']:
            for breed in readQuery['breed']['$in']:
                if breed_min > breed:
                    readQuery['breed']['$in'].remove(breed)
        # No breed changes made from radio button
        else:
            readQuery['breed']['$gte'] = breed_min
    # breed_max changes
    if breed_max != '':
        if '$in' in readQuery['breed']:
            for breed in readQuery['breed']['$in']:
                if breed_max < breed:
                    readQuery['breed']['$in'].remove(breed)
        # No breed changes made from radio button
        else:
            readQuery['breed']['$lte'] = breed_max

    # age_upon_outcome_in_weeks changes
    if (age_min.isdigit() or age_max.isdigit()) and 'age_upon_outcome_in_weeks' not in readQuery:
        readQuery['age_upon_outcome_in_weeks'] = {}
    # age_min changes
    if age_min.isdigit():
        # Replace the minimum age if one currently doesn't exist, or if the current one is higher than the range value
        if '$gte' not in readQuery['age_upon_outcome_in_weeks'] or int(age_min) > readQuery['age_upon_outcome_in_weeks']['$gte']:
            readQuery['age_upon_outcome_in_weeks']['$gte'] = int(age_min)
    # Check if number necessity failure occurred    
    elif age_min != '':
        numberNecessityFailure = True
    # age_max changes
    if age_max.isdigit():
        # Replace the minimum age if one currently doesn't exist, or if the current one is lower than the range value
        if '$lte' not in readQuery['age_upon_outcome_in_weeks'] or int(age_max) < readQuery['age_upon_outcome_in_weeks']['$lte']:
            readQuery['age_upon_outcome_in_weeks']['$lte'] = int(age_max)
    # Check if number necessity failure occurred    
    elif age_max != '':
        numberNecessityFailure = True

    # name changes
    if name_min != '' or name_max != '':
        readQuery['name'] = {}
        if name_min != '':
            readQuery['name']['$gte'] = name_min
        if name_max != '':
            readQuery['name']['$lte'] = name_max

    # location_lat changes
    # Check that at least one of the values can be a float before instantiating location_lat in readQuery
    try:
        float(lat_min)
    except:
        # Check if number necessity failure occurred
        if lat_min != '':
            numberNecessityFailure = True
        try: 
            float(lat_max)
        except:
            # Check if number necessity failure occurred
            if lat_max != '':
                numberNecessityFailure = True
        # Value can be a float
        else:
            readQuery['location_lat'] = {}
            # Assign lat_max: we know it's a float
            readQuery['location_lat']['$lte'] = float(lat_max)
    # Value can be a float
    else:
        readQuery['location_lat'] = {}
        # Assign lat_min: we know it's a float
        readQuery['location_lat']['$gte'] = float(lat_min)

        # Try to assign lat_max (other case already taken care of in except)
        try: 
            readQuery['location_lat']['$lte'] = float(lat_max)
        except:
            # Check if number necessity failure occurred
            if lat_max != '':
                numberNecessityFailure = True

    # location_long changes
    # Check that at least one of the values can be a float before instantiating location_lat in readQuery
    try:
        float(long_min)
    except:
        # Check if number necessity failure occurred
        if long_min != '':
            numberNecessityFailure = True
        try: 
            float(long_max)
        except:
            # Check if number necessity failure occurred
            if long_max != '':
                numberNecessityFailure = True
        # Value can be a float
        else:
            readQuery['location_long'] = {}
            # Assign long_max: we know it's a float
            readQuery['location_long']['$lte'] = float(long_max)
    # Value can be a float
    else:
        readQuery['location_long'] = {}
        # Assign long_min: we know it's a float
        readQuery['location_long']['$gte'] = float(long_min)

        # Try to assign lat_max (other case already taken care of in except)
        try: 
            readQuery['location_long']['$lte'] = float(long_max)
        except:
            # Check if number necessity failure occurred
            if long_max != '':
                numberNecessityFailure = True
    
    # Check if any changes were made to the readQuery
    if readQuery:
        # Get the data from the final read query
        data = list(db.read(readQuery))
    # No changes were made to the readQuery
    else:
        # Update data frame before using it in case changes were made
        global df
        df = pd.DataFrame.from_records(db.read({}))
        data = df.to_dict('records')
    
    columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]

    return data, columns, not numberNecessityFailure

# Function to open the create table entry page
@callback(
    Output('url', 'pathname', allow_duplicate = True),
    Output('create-table-entry-clicks', 'data'),
    Input('create-entry-button', 'n_clicks'),
    State('create-table-entry-clicks', 'data'),
    prevent_initial_call = True
)
def open_create_table_entry(create_table_entry_clicks, stored_clicks):
    if create_table_entry_clicks is None or create_table_entry_clicks < stored_clicks:
        return no_update, stored_clicks
    
    return '/create', stored_clicks + 1

# Function to open the update table entry page
@callback(
    Output('url', 'pathname', allow_duplicate = True),
    Output('update-table-entry-clicks', 'data'),
    Input('update-entry-button', 'n_clicks'),
    State('update-table-entry-clicks', 'data'),
    prevent_initial_call = True
)
def open_update_table_entry(update_table_entry_clicks, stored_clicks):
    if update_table_entry_clicks is None or update_table_entry_clicks < stored_clicks:
        return no_update, stored_clicks
    
    return '/update', stored_clicks + 1

# Function to open the delete table entry page
@callback(
    Output('url', 'pathname', allow_duplicate = True),
    Output('delete-table-entry-clicks', 'data'),
    Input('delete-entry-button', 'n_clicks'),
    State('delete-table-entry-clicks', 'data'),
    prevent_initial_call = True
)
def open_delete_table_entry(delete_table_entry_clicks, stored_clicks):
    if delete_table_entry_clicks is None or delete_table_entry_clicks < stored_clicks:
        return no_update, stored_clicks
    
    return '/delete', stored_clicks + 1

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    Input('datatable-id', "derived_virtual_data"),
    Input('filter-type', 'value'))
def update_graphs(viewData, filter_type):
    dff = pd.DataFrame.from_dict(viewData)

    # Ensure that the DataFrame is not empty
    if not dff.empty:
        # DataFrame has data, display pie chart
        return [
            dcc.Graph(            
                figure = px.pie(dff, names = 'breed', title = 'Preferred Animals')
            )    
        ]
    # DataFrame is empty, remove pie chart
    else:
        return None
        
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    Input('datatable-id', 'selected_columns')
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    Input('datatable-id', "derived_virtual_data"),
    Input('datatable-id', "derived_virtual_selected_rows"))
def update_map(viewData, index):
    if viewData is None:
        return
    elif index == [] or index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)

    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else:
        row = index[0]

    # Columns 6 and 7 define the grid-coordinates for the map
    location_lat = 6
    location_long = 7
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(
            style = {
                'width': '800px', 
                'height': '500px'
            },
            # Center map at marker coordinates
            center = [dff.iloc[row, location_lat], dff.iloc[row, location_long]], 
            zoom = 10, 
            children=[
                dl.TileLayer(id = "base-layer-id"),
                # Marker with tool tip and popup
                # Column 2 defines the breed for the animal
                # Column 5 defines the name of the animal
                dl.Marker(position = [dff.iloc[row, location_lat], dff.iloc[row, location_long]], children = [
                    dl.Tooltip(dff.iloc[row, 2]),
                    dl.Popup([
                        html.P("Animal Name:"),
                        html.H1(dff.iloc[row, 5])
                    ])
                ])
            ]
        )
    ]

# Function to log the user out
@callback(
    Output('url', 'pathname', allow_duplicate = True),
    Output('logout-clicks', 'data'),
    Input('logout-button', 'n_clicks'),
    State('logout-clicks', 'data'),
    prevent_initial_call = True
)
def log_user_out(logout_clicks, stored_clicks):
    if logout_clicks is None or logout_clicks < stored_clicks:
        return no_update, stored_clicks

    # Only logout on an actual click
    logout_user()
    
    return '/login', stored_clicks + 1

#########################
# Table Entry Creation Layout / View
#########################

# Create table entry page
create_table_entry_layout = html.Div([
    html.Div(
        className='col s12 m6',
        children = [
            html.Center(html.Img(
                src = 'data:image/png;base64,{}'.format(encoded_image.decode()), 
                style = {
                    'height': '5%', 
                    'width': '5%'
                }
            ))
        ]
    ),
    html.Center(html.B(html.H1('Create Table Entry'))),
    html.Center(html.B(html.H3(
        'animal_type', 
        style = {
            'marginTop': 1,
            'marginBottom': 0
        }
    ))),
    html.Center(html.Div(
        className='col s12 m6',
        children = [
            dcc.RadioItems(
                id = 'create-animal-type-input',
                options = [
                    {'label': 'Cat', 'value': 'Cat'},
                    {'label': 'Dog', 'value': 'Dog'}
                ],
                value = 'Cat',
                inline = True,
                labelStyle = {'margin-left': 10, 'margin-right': 10, 'align-items': 'center'},
                style = {'marginTop': 5}
            )
        ]
    )),
    html.Center(html.B(html.H3(
        'breed', 
        style = {
            'marginTop': 15,
            'marginBottom': 0
        }
    ))),
    html.Center(html.Div(
        dcc.Input(
            id = 'create-breed-text', 
            placeholder = 'Add breed', 
            value = '',
            style = {
                'width': '10%',
                'marginTop': 5,
                'marginBottom': 0
            }
        )
    )),
    html.Center(html.B(html.H3(
        'sex_upon_outcome', 
        style = {
            'marginTop': 15,
            'marginBottom': 0
        }
    ))),
    html.Center(html.Div(
        className='col s12 m6',
        children = [
            dcc.RadioItems(
                id = 'create-sex-upon-outcome-input',
                options = [
                    {'label': 'Intact Male', 'value': 'Intact Male'},
                    {'label': 'Neutered Male', 'value': 'Neutered Male'},
                    {'label': 'Intact Female', 'value': 'Intact Female'},
                    {'label': 'Spayed Female', 'value': 'Spayed Female'}
                ],
                value = 'Intact Male',
                inline = True,
                labelStyle = {'margin-left': 10, 'margin-right': 10, 'align-items': 'center'},
                style = {'marginTop': 5}
            )
        ]
    )),
    html.Center(html.B(html.H3(
        'age_upon_outcome_in_weeks', 
        style = {
            'marginTop': 15,
            'marginBottom': 0
        }
    ))),
    html.Center(html.Div(
        dcc.Input(
            id = 'create-age-upon-outcome-text', 
            placeholder = 'Add age upon outcome', 
            value = '',
            style = {
                'width': '10%',
                'marginTop': 5,
                'marginBottom': 0
            }
        )
    )),
    html.Center(html.B(html.H3(
        'Age must be a positive whole number', 
        id = 'create-bad-age-text', 
        style = {
            'color': 'red',
            'marginTop': 5,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Center(html.B(html.H3(
        'name', 
        style = {
            'marginTop': 15,
            'marginBottom': 0
        }
    ))),
    html.Center(html.Div(
        dcc.Input(
            id = 'create-name-text', 
            placeholder = 'Add name', 
            value = '',
            style = {
                'width': '10%',
                'marginTop': 5,
                'marginBottom': 0
            }
        )
    )),
    html.Center(html.B(html.H3(
        'location_lat', 
        style = {
            'marginTop': 15,
            'marginBottom': 0
        }
    ))),
    html.Center(html.Div(
        dcc.Input(
            id = 'create-location-lat-text', 
            placeholder = 'Add location (latitude)', 
            value = '',
            style = {
                'width': '10%',
                'marginTop': 5,
                'marginBottom': 0
            }
        )
    )),
    html.Center(html.B(html.H3(
        'Location values must be numbers larger than -180 and smaller than 180', 
        id = 'create-bad-location-lat-text', 
        style = {
            'color': 'red',
            'marginTop': 5,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Center(html.B(html.H3(
        'location_long', 
        style = {
            'marginTop': 15,
            'marginBottom': 0
        }
    ))),
    html.Center(html.Div(
        dcc.Input(
            id = 'create-location-long-text', 
            placeholder = 'Add location (longitude)', 
            value = '',
            style = {
                'width': '10%',
                'marginTop': 5,
                'marginBottom': 0
            }
        )
    )),
    html.Center(html.B(html.H3(
        'Location values must be numbers larger than -180 and smaller than 180', 
        id = 'create-bad-location-long-text', 
        style = {
            'color': 'red',
            'marginTop': 5,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Div(
        # Make child Divs go next to each other
        className='row',
        style = {
            'display': 'flex', 
            'justifyContent': 'center', 
            'marginTop': 25
        },
        children = [   
            html.Button(
                id = 'create-button', 
                children = 'Create table entry', 
                style = {'backgroundColor': '#67bed9'},
                n_clicks = 0
            ),
            html.Button(
                id = 'cancel-create-button', 
                children = 'Cancel',
                style = {'backgroundColor': '#d07597', 'marginLeft': 15},
                n_clicks = 0
            )
        ]
    ),
    html.Center(html.B(html.H3(
        'Creating a new entry requires all columns have a value', 
        id = 'bad-create-input-text', 
        style = {
            'color': 'red',
            'marginTop': 5,
            'marginBottom': 0
        }, 
        hidden = True
    )))
])

#############################################
# Interaction Between Table Entry Creation Components / Controller
#############################################

# Function to return to the database page from the Create page
@callback(
    Output('url', 'pathname', allow_duplicate = True),
    Output('create-cancel-clicks', 'data'),
    Input('cancel-create-button', 'n_clicks'),
    State('create-cancel-clicks', 'data'),
    prevent_initial_call = True
)
def cancel_create(create_cancel_clicks, stored_clicks):
    if create_cancel_clicks is None or create_cancel_clicks < stored_clicks:
        return no_update, stored_clicks
    
    return '/database', stored_clicks + 1

# Function for attempting to create a database entry
@callback(
    Output('url', 'pathname', allow_duplicate = True),
    Output('create-clicks', 'data'),
    Output('create-bad-age-text', 'hidden'),
    Output('create-bad-location-lat-text', 'hidden'),
    Output('create-bad-location-long-text', 'hidden'),
    Output('bad-create-input-text', 'hidden'),
    Input('create-button', 'n_clicks'),
    State('create-clicks', 'data'),
    State('create-animal-type-input', 'value'),
    State('create-breed-text', 'value'),
    State('create-sex-upon-outcome-input', 'value'),
    State('create-age-upon-outcome-text', 'value'),
    State('create-name-text', 'value'),
    State('create-location-lat-text', 'value'),
    State('create-location-long-text', 'value'),
    prevent_initial_call = True
)
def try_create_entry(create_clicks, stored_clicks, animal_type_input,
                     breed_input, sex_input, age_input, name_input,
                     location_lat_input, location_long_input):
    if create_clicks is None or create_clicks < stored_clicks:
        return no_update, stored_clicks, True, True, True, True    

    # Create variables for the potential issues
    badValues = False
    badAge, badLat, badLong = check_numerical_columns(age_input, location_lat_input, location_long_input)

    # Check that all columns have values
    if breed_input == '' or age_input == '' or name_input == '' or location_lat_input == '' or location_long_input == '':
        badValues = True
            
    if not badAge and not badLat and not badLong and not badValues:
        creationValues = { 'animal_type': animal_type_input, 'breed': breed_input, 'sex_upon_outcome': sex_input, 
                           'age_upon_outcome_in_weeks': int(age_input), 'name': name_input, 'location_lat': float(location_lat_input),
                           'location_long': float(location_long_input) }
        db.create(creationValues)
        return '/database', stored_clicks + 1, True, True, True, True
    else:
        return no_update, stored_clicks + 1, not badAge, not badLat, not badLong, not badValues

#########################
# Table Entry Update Layout / View
#########################

# Update table entry page
update_table_entry_layout = html.Div([
    html.Div(
        className='col s12 m6',
        children = [
            html.Center(html.Img(
                src = 'data:image/png;base64,{}'.format(encoded_image.decode()), 
                style = {
                    'height': '5%', 
                    'width': '5%'
                }
            ))
        ]
    ),
    html.Center(html.B(html.H1('Update Table Entry'))),
    html.Center(html.B(html.H2('Enter animal_id of entry to update:'))),
    html.Center(html.Div(
        dcc.Input(
            id = 'update-id-input', 
            placeholder = 'Add animal_id (ex: A00007)', 
            value = '',
            style = {
                'width': '15%',
                'marginTop': 0
            }
        )
    )),
    html.Center(html.Button(
        id = 'find-update-button', 
        children = 'Find',
        style = {'backgroundColor': '#67bed9', 'marginTop': 10},
        n_clicks = 0
    )),
    html.Center(html.B(html.H3(
        'Entry with that animal_id does not exist', 
        id = 'bad-update-find-text', 
        style = {
            'color': 'red',
            'marginTop': 5,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Center(html.B(html.H3(
        'animal_id',
        id = 'update-animal-id',
        style = {
            'marginTop': 5,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Center(html.B(html.H3(
        'animal_type',
        id = 'update-animal-type-text', 
        style = {
            'marginTop': 15,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Center(html.Div(
        id = 'update-animal-type-input-div',
        className = 'col s12 m6',
        children = [
            dcc.RadioItems(
                id = 'update-animal-type-input',
                options = [
                    {'label': 'Cat', 'value': 'Cat'},
                    {'label': 'Dog', 'value': 'Dog'}
                ],
                value = 'Cat',
                inline = True,
                labelStyle = {'margin-left': 10, 'margin-right': 10, 'align-items': 'center'},
                style = {'marginTop': 5}
            )
        ], 
        hidden = True
    )),
    html.Center(html.B(html.H3(
        'breed',
        id = 'update-breed-text',
        style = {
            'marginTop': 15,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Center(html.Div(
        id = 'update-breed-input-div', 
        children = [   
            dcc.Input(
                id = 'update-breed-input', 
                placeholder = 'Add breed', 
                value = '',
                style = {
                    'width': '10%',
                    'marginTop': 5,
                    'marginBottom': 0
                }
            )
        ], 
        hidden = True
    )),
    html.Center(html.B(html.H3(
        'sex_upon_outcome',
        id = 'update-sex-text', 
        style = {
            'marginTop': 15,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Center(html.Div(
        id = 'update-sex-input-div',
        className = 'col s12 m6',
        children = [
            dcc.RadioItems(
                id = 'update-sex-input',
                options = [
                    {'label': 'Intact Male', 'value': 'Intact Male'},
                    {'label': 'Neutered Male', 'value': 'Neutered Male'},
                    {'label': 'Intact Female', 'value': 'Intact Female'},
                    {'label': 'Spayed Female', 'value': 'Spayed Female'}
                ],
                value = 'Intact Male',
                inline = True,
                labelStyle = {'margin-left': 10, 'margin-right': 10, 'align-items': 'center'},
                style = {'marginTop': 5}
            )
        ], 
        hidden = True
    )),
    html.Center(html.B(html.H3(
        'age_upon_outcome_in_weeks',
        id = 'update-age-text', 
        style = {
            'marginTop': 15,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Center(html.Div(
        id = 'update-age-input-div', 
        children = [   
            dcc.Input(
                id = 'update-age-input', 
                placeholder = 'Add age upon outcome', 
                value = '',
                style = {
                    'width': '10%',
                    'marginTop': 5,
                    'marginBottom': 0
                }
            )
        ], 
        hidden = True
    )),
    html.Center(html.B(html.H3(
        'Age must be a positive whole number', 
        id = 'update-bad-age-text', 
        style = {
            'color': 'red',
            'marginTop': 5,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Center(html.B(html.H3(
        'name',
        id = 'update-name-text', 
        style = {
            'marginTop': 15,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Center(html.Div(
        id = 'update-name-input-div', 
        children = [   
            dcc.Input(
                id = 'update-name-input', 
                placeholder = 'Add name', 
                value = '',
                style = {
                    'width': '10%',
                    'marginTop': 5,
                    'marginBottom': 0
                }
            )
        ], 
        hidden = True
    )),
    html.Center(html.B(html.H3(
        'location_lat', 
        id = 'update-location-lat-text',
        style = {
            'marginTop': 15,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Center(html.Div(
        id = 'update-location-lat-input-div', 
        children = [   
            dcc.Input(
                id = 'update-location-lat-input', 
                placeholder = 'Add location (latitude)', 
                value = '',
                style = {
                    'width': '10%',
                    'marginTop': 5,
                    'marginBottom': 0
                }
            )
        ], 
        hidden = True
    )),
    html.Center(html.B(html.H3(
        'Location values must be numbers larger than -180 and smaller than 180', 
        id = 'update-bad-location-lat-text', 
        style = {
            'color': 'red',
            'marginTop': 5,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Center(html.B(html.H3(
        'location_long',
        id = 'update-location-long-text', 
        style = {
            'marginTop': 15,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Center(html.Div(
        id = 'update-location-long-input-div', 
        children = [   
            dcc.Input(
                id = 'update-location-long-input', 
                placeholder = 'Add location (longitude)', 
                value = '',
                style = {
                    'width': '10%',
                    'marginTop': 5,
                    'marginBottom': 0
                }
            )
        ], 
        hidden = True
    )),
    html.Center(html.B(html.H3(
        'Location values must be numbers larger than -180 and smaller than 180', 
        id = 'update-bad-location-long-text', 
        style = {
            'color': 'red',
            'marginTop': 5,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Div(
        # Make child Divs go next to each other
        className='row',
        style = {
            'display': 'flex', 
            'justifyContent': 'center', 
            'marginTop': 25
        },
        children = [   
            html.Button(
                id = 'update-button', 
                children = 'Update table entry', 
                style = {'backgroundColor': '#9c9ab8'},
                n_clicks = 0
            ),
            html.Button(
                id = 'cancel-update-button', 
                children = 'Cancel',
                style = {'backgroundColor': '#d07597', 'marginLeft': 15},
                n_clicks = 0
            )
        ]
    ),
    html.Center(html.B(html.H3(
        'Entry must be found before attempting an update operation', 
        id = 'no-update-input-text', 
        style = {
            'color': 'red',
            'marginTop': 5,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Center(html.B(html.H3(
        'Updating an entry requires all columns have a value', 
        id = 'update-missing-column-text', 
        style = {
            'color': 'red',
            'marginTop': 5,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Center(html.B(html.H3(
        'Updating a new entry requires at least one column have a new value', 
        id = 'bad-update-input-text', 
        style = {
            'color': 'red',
            'marginTop': 5,
            'marginBottom': 0
        }, 
        hidden = True
    )))
])

#############################################
# Interaction Between Table Entry Update Components / Controller
#############################################

# Function to have the delete page display the information of the entry about to be deleted
@callback(
    Output('bad-update-find-text', 'hidden'),
    Output('update-animal-id', 'hidden'),
    Output('update-animal-id', 'children'),
    Output('update-animal-type-text', 'hidden'),
    Output('update-animal-type-input-div', 'hidden'),
    Output('update-animal-type-input', 'value'),
    Output('update-breed-text', 'hidden'),
    Output('update-breed-input-div', 'hidden'),
    Output('update-breed-input', 'value'),
    Output('update-sex-text', 'hidden'),
    Output('update-sex-input-div', 'hidden'),
    Output('update-sex-input', 'value'),
    Output('update-age-text', 'hidden'),
    Output('update-age-input-div', 'hidden'),
    Output('update-age-input', 'value'),
    Output('update-name-text', 'hidden'),
    Output('update-name-input-div', 'hidden'),
    Output('update-name-input', 'value'),
    Output('update-location-lat-text', 'hidden'),
    Output('update-location-lat-input-div', 'hidden'),
    Output('update-location-lat-input', 'value'),
    Output('update-location-long-text', 'hidden'),
    Output('update-location-long-input-div', 'hidden'),
    Output('update-location-long-input', 'value'),
    Input('find-update-button', 'n_clicks'),
    State('update-id-input', 'value'),
    prevent_initial_call = True
)
def try_find_update(update_find_clicks, input_id):
    if input_id == '':
        return True, True, no_update, True, True, no_update, True, True, no_update, True, True, no_update, True, True, no_update, True, True, no_update, True, True, no_update, True, True, no_update

    if db.doesEntryExist(input_id):
        foundEntry = db.read({'animal_id': input_id})[0]
        animalIdInfo = 'animal_id: ' + foundEntry['animal_id']
        animalTypeInfo = foundEntry['animal_type']
        breedInfo = foundEntry['breed']
        sexInfo = foundEntry['sex_upon_outcome']
        ageInfo = str(foundEntry['age_upon_outcome_in_weeks'])
        nameInfo = foundEntry['name']
        locationLatInfo = str(foundEntry['location_lat'])
        locationLongInfo = str(foundEntry['location_long'])
        return True, False, animalIdInfo, False, False, animalTypeInfo, False, False, breedInfo, False, False, sexInfo, False, False, ageInfo, False, False, nameInfo, False, False, locationLatInfo, False, False, locationLongInfo
    else:
        return False, True, no_update, True, True, no_update, True, True, no_update, True, True, no_update, True, True, no_update, True, True, no_update, True, True, no_update, True, True, no_update

# Function to return to the database page from the Update page
@callback(
    Output('url', 'pathname', allow_duplicate = True),
    Output('update-cancel-clicks', 'data'),
    Input('cancel-update-button', 'n_clicks'),
    State('update-cancel-clicks', 'data'),
    prevent_initial_call = True
)
def cancel_update(update_cancel_clicks, stored_clicks):
    if update_cancel_clicks is None or update_cancel_clicks < stored_clicks:
        return no_update, stored_clicks
    
    return '/database', stored_clicks + 1

# WORKSPACE: Function for attempting to update a database entry
@callback(
    Output('url', 'pathname', allow_duplicate = True),
    Output('update-clicks', 'data'),
    Output('no-update-input-text', 'hidden'),
    Output('update-bad-age-text', 'hidden'),
    Output('update-bad-location-lat-text', 'hidden'),
    Output('update-bad-location-long-text', 'hidden'),
    Output('update-missing-column-text', 'hidden'),
    Output('bad-update-input-text', 'hidden'),
    Input('update-button', 'n_clicks'),
    State('update-clicks', 'data'),
    State('update-animal-id', 'children'),
    State('update-animal-id', 'hidden'),
    State('update-animal-type-input', 'value'),
    State('update-breed-input', 'value'),
    State('update-sex-input', 'value'),
    State('update-age-input', 'value'),
    State('update-name-input', 'value'),
    State('update-location-lat-input', 'value'),
    State('update-location-long-input', 'value'),
    prevent_initial_call = True
)
def try_update(update_clicks, stored_clicks, animal_id_string, animal_not_found, input_animal_type,
              input_breed, input_sex, input_age, input_name, input_location_lat, 
              input_location_long):
    if update_clicks is None or update_clicks < stored_clicks:
        return no_update, stored_clicks, True, True, True, True, True, True

    # Catch case of no animal being found
    if animal_not_found:
        return no_update, stored_clicks + 1, False, True, True, True, True, True

    # Catch case of a non-number existing in a numerical column
    badAge, badLat, badLong = check_numerical_columns(input_age, input_location_lat, input_location_lat)

    # Catch case of an empty column existing
    emptyColumnExists = False
    if input_breed == '' or input_age == '' or input_name == '' or input_location_lat == '' or input_location_long == '':
        emptyColumnExists = True

    # Stop attempt here if any previous issue exists
    if badAge or badLat or badLong or emptyColumnExists:
        return no_update, stored_clicks + 1, True, not badAge, not badLat, not badLong, not emptyColumnExists, True

    # Catch case of no new changes being made
    foundEntry = db.read({'animal_id': animal_id_string[-6:]})[0]
    animalTypeInfo = foundEntry['animal_type']
    breedInfo = foundEntry['breed']
    sexInfo = foundEntry['sex_upon_outcome']
    ageInfo = str(foundEntry['age_upon_outcome_in_weeks'])
    nameInfo = foundEntry['name']
    locationLatInfo = str(foundEntry['location_lat'])
    locationLongInfo = str(foundEntry['location_long'])
    if input_animal_type == animalTypeInfo and input_breed == breedInfo and input_sex == sexInfo and input_age == ageInfo and input_name == nameInfo and input_location_lat == locationLatInfo and input_location_long == locationLongInfo:
        return no_update, stored_clicks + 1, True, True, True, True, True, False

    # Update table entry with changes
    entryChanges = {
        'animal_type': input_animal_type,
        'breed': input_breed,
        'sex_upon_outcome': input_sex,
        'age_upon_outcome_in_weeks': int(input_age),
        'name': input_name,
        'location_lat': float(input_location_lat),
        'location_long': float(input_location_long)
    }
    db.update(animal_id_string[-6:], entryChanges)
    
    return '/database', stored_clicks + 1, True, True, True, True, True, True

#########################
# Table Entry Deletion Layout / View
#########################

delete_table_entry_layout = html.Div([
    html.Div(
        className='col s12 m6',
        children = [
            html.Center(html.Img(
                src = 'data:image/png;base64,{}'.format(encoded_image.decode()), 
                style = {
                    'height': '5%', 
                    'width': '5%'
                }
            ))
        ]
    ),
    html.Center(html.B(html.H1('Delete Table Entry'))),
    html.Center(html.B(html.H2('Enter animal_id of entry to delete:'))),
    html.Center(html.Div(
        dcc.Input(
            id = 'delete-id-input', 
            placeholder = 'Add animal_id (ex: A00007)', 
            value = '',
            style = {
                'width': '15%',
                'marginTop': 0
            }
        )
    )),
    html.Center(html.Button(
        id = 'find-delete-button', 
        children = 'Find',
        style = {'backgroundColor': '#67bed9', 'marginTop': 10},
        n_clicks = 0
    )),
    html.Center(html.B(html.H3(
        'Entry with that animal_id does not exist', 
        id = 'bad-delete-input-text', 
        style = {
            'color': 'red',
            'marginTop': 5,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Center(html.B(html.H3(
        'animal_id',
        id = 'delete-animal-id',
        style = {
            'marginTop': 5,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Center(html.B(html.H3(
        'animal_type',
        id = 'delete-animal-type',
        style = {
            'marginTop': 15,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Center(html.B(html.H3(
        'breed', 
        id = 'delete-breed',
        style = {
            'marginTop': 15,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Center(html.B(html.H3(
        'sex_upon_outcome',
        id = 'delete-sex',
        style = {
            'marginTop': 15,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Center(html.B(html.H3(
        'age_upon_outcome_in_weeks',
        id = 'delete-age',
        style = {
            'marginTop': 15,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Center(html.B(html.H3(
        'name', 
        id = 'delete-name',
        style = {
            'marginTop': 15,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Center(html.B(html.H3(
        'location_lat',
        id = 'delete-location-lat',
        style = {
            'marginTop': 15,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Center(html.B(html.H3(
        'location_long',
        id = 'delete-location-long',
        style = {
            'marginTop': 15,
            'marginBottom': 0
        }, 
        hidden = True
    ))),
    html.Div(
        # Make child Divs go next to each other
        className='row',
        style = {
            'display': 'flex', 
            'justifyContent': 'center', 
            'marginTop': 25
        },
        children = [   
            html.Button(
                id = 'delete-button', 
                children = 'Delete table entry', 
                style = {'backgroundColor': '#d07597'},
                n_clicks = 0
            ),
            html.Button(
                id = 'cancel-delete-button', 
                children = 'Cancel',
                style = {'backgroundColor': '#67bed9', 'marginLeft': 15},
                n_clicks = 0
            )
        ]
    ),
    html.Center(html.B(html.H3(
        'Entry must be found before attempting a deletion operation', 
        id = 'no-delete-input-text', 
        style = {
            'color': 'red',
            'marginTop': 5,
            'marginBottom': 0
        }, 
        hidden = True
    )))
])

#############################################
# Interaction Between Table Entry Deletion Components / Controller
#############################################

# Function to have the delete page display the information of the entry about to be deleted
@callback(
    Output('bad-delete-input-text', 'hidden'),
    Output('delete-animal-id', 'children'),
    Output('delete-animal-id', 'hidden'),
    Output('delete-animal-type', 'children'),
    Output('delete-animal-type', 'hidden'),
    Output('delete-breed', 'children'),
    Output('delete-breed', 'hidden'),
    Output('delete-sex', 'children'),
    Output('delete-sex', 'hidden'),
    Output('delete-age', 'children'),
    Output('delete-age', 'hidden'),
    Output('delete-name', 'children'),
    Output('delete-name', 'hidden'),
    Output('delete-location-lat', 'children'),
    Output('delete-location-lat', 'hidden'),
    Output('delete-location-long', 'children'),
    Output('delete-location-long', 'hidden'),
    Input('find-delete-button', 'n_clicks'),
    State('delete-id-input', 'value'),
    prevent_initial_call = True
)
def try_find_delete(delete_find_clicks, input_id):
    if input_id == '':
        return True, no_update, True, no_update, True, no_update, True, no_update, True, no_update, True, no_update, True, no_update, True, no_update, True

    if db.doesEntryExist(input_id):
        foundEntry = db.read({'animal_id': input_id})[0]
        animalIdInfo = 'animal_id: ' + foundEntry['animal_id']
        animalTypeInfo = 'animal_type: ' + foundEntry['animal_type']
        breedInfo = 'breed: ' + foundEntry['breed']
        sexInfo = 'sex_upon_outcome: ' + foundEntry['sex_upon_outcome']
        ageInfo = 'age_upon_outcome_in_weeks: ' + str(foundEntry['age_upon_outcome_in_weeks'])
        nameInfo = 'name: ' + foundEntry['name']
        locationLatInfo = 'location_lat: ' + str(foundEntry['location_lat'])
        locationLongInfo = 'location_long: ' + str(foundEntry['location_long'])
        return True, animalIdInfo, False, animalTypeInfo, False, breedInfo, False, sexInfo, False, ageInfo, False, nameInfo, False, locationLatInfo, False, locationLongInfo, False
    else:
        return False, no_update, True, no_update, True, no_update, True, no_update, True, no_update, True, no_update, True, no_update, True, no_update, True
    
# Function to return to the database page from the Delete page
@callback(
    Output('url', 'pathname', allow_duplicate = True),
    Output('delete-cancel-clicks', 'data'),
    Input('cancel-delete-button', 'n_clicks'),
    State('delete-cancel-clicks', 'data'),
    prevent_initial_call = True
)
def cancel_delete(delete_cancel_clicks, stored_clicks):
    if delete_cancel_clicks is None or delete_cancel_clicks < stored_clicks:
        return no_update, stored_clicks
    
    return '/database', stored_clicks + 1

# Function for attempting to delete a database entry
@callback(
    Output('url', 'pathname', allow_duplicate = True),
    Output('delete-clicks', 'data'),
    Output('no-delete-input-text', 'hidden'),
    Input('delete-button', 'n_clicks'),
    State('delete-clicks', 'data'),
    State('delete-animal-id', 'children'),
    State('delete-animal-id', 'hidden'),
    prevent_initial_call = True
)
def try_delete(delete_clicks, stored_clicks, animal_id_string, animal_not_found):
    if delete_clicks is None or delete_clicks < stored_clicks:
        return no_update, stored_clicks, True

    # Catch case of no animal being found
    if animal_not_found:
        return no_update, stored_clicks + 1, False

    # Get the animal_id of the found entry
    foundId = animal_id_string[-6:]
    # Delete the found entry
    db.delete(foundId)
    
    return '/database', stored_clicks + 1, True

#############################################
# Multipurpose Functions
#############################################

# Function to check validity of numerical column values
def check_numerical_columns(age_input, location_lat_input, location_long_input):
    # Create variables for the potential issues
    badAge = False
    badLat = False
    badLong = False
    
    # Check to ensure that all values that need to be numbers fit their needs
    if not age_input.isdigit() or int(age_input) < 0:
        badAge = True
    
    try:
        float(location_lat_input)
    except:
        badLat = True
    else:
        if float(location_lat_input) < -180 or float(location_lat_input) > 180:
            badLat = True

    try:
        float(location_long_input)
    except:
        badLong = True
    else:
        if float(location_long_input) < -180 or float(location_long_input) > 180:
            badLong = True

    return badAge, badLat, badLong

# Run app and display result in jupyterlab mode
# If you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server()

Dash is running on http://127.0.0.1:8050/

Dash app running on http://127.0.0.1:8050/
